# Signal construction

Builds the four pre-registered signals and looks at how they relate to each other and to the controls.
Only same-day and past data is used here; forward returns are not touched until the evaluation notebook.

In [1]:
import numpy as np
import pandas as pd

from shortvol import features, finra, prices

pd.set_option("display.width", 140)
panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
df = features.add_all(panel, days)

## 1. Shrinkage constant V0 (estimated on 2016-2019 only)
Within a stock, SVR5 wobbles around its own baseline.
Part of that is real variation (s2), and part is sampling noise that shrinks with volume (c / V).
Across 20 volume buckets we fit `variance = s2 + c / V`.
V0 = c / s2 is the 5-day volume at which noise equals real variation, and the weight on today's value is V / (V + V0).

Clarification of the pre-registration wording: "real variation" is measured around the stock's own average, because that is what the signal is shrunk toward.
It is still estimated from short volume alone, with no returns.

In [2]:
fit = features.estimate_v0(df)
V0 = fit["v0"]
print(f"V0 = {V0:,.0f} shares over 5 days   (s2 = {fit['s2']:.4f}, c = {fit['c']:.1f})")
fit["buckets"].assign(fitted=fit["s2"] + fit["c"] * fit["buckets"].inv_v).round(6)

V0 = 63,562 shares over 5 days   (s2 = 0.0097, c = 617.0)


,var,inv_v,fitted
bucket,,,
0,0.010311,0.000003,0.011386
1,0.010747,0.000002,0.010658
2,0.010704,0.000001,0.010432
3,0.010813,0.000001,0.010297
4,0.010811,0.000001,0.010204
5,0.010620,0.000001,0.010135
6,0.010739,0.000001,0.010080
7,0.010401,0.000001,0.010033
8,0.010365,0.000000,0.009992


Variance barely changes with volume: the fitted noise term is small next to the real variation.
In the S&P 500 even the thinnest names trade enough off-exchange for SVR5 to be precise.

In [3]:
df = features.add_all(panel, days, v0=V0)
w = df.tv5 / (df.tv5 + V0)
print(w.describe(percentiles=[0.01, 0.05, 0.5]).round(3).to_string())
print(f"\nmedian 5-day off-exchange volume: {df.tv5.median():,.0f} shares")

count    1306903.000
mean           0.967
std            0.038
min            0.468
1%             0.815
5%             0.894
50%            0.979
max            1.000

median 5-day off-exchange volume: 3,000,197 shares


## 2. How different are the four signals?
Average daily cross-sectional rank correlation between each pair.
Shrunk SVR5 is nearly identical to SVR5, so in practice there are three distinct signals.

In [4]:
SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]

def daily_rank_corr(df, cols):
    ranks = df.groupby("date")[cols].rank()
    return ranks.groupby(df.date).corr().groupby(level=1).mean().loc[cols, cols]

daily_rank_corr(df, SIGNALS).round(3)

,svr5,svr1,svr_abnormal,svr5_shrunk
svr5,1.000,0.737,0.708,0.999
svr1,0.737,1.000,0.507,0.737
svr_abnormal,0.708,0.507,1.000,0.695
svr5_shrunk,0.999,0.737,0.695,1.000


## 3. What the signal moves with on the same day
Average daily rank correlation of each signal with the controls that neutralization will remove.
Thesis B says dealers short into customer buying, so SVR should be higher on up days (positive with `ret1`).
This is the same-day pattern SqueezeMetrics reported; it says nothing yet about tomorrow's return.

In [5]:
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume", "off_exchange_share"]
daily_rank_corr(df, SIGNALS + CONTROLS).loc[SIGNALS, CONTROLS].round(3)

,ret1,ret5,log_adv20,abnormal_volume,off_exchange_share
svr5,0.016,0.110,-0.216,-0.028,-0.073
svr1,0.088,0.134,-0.187,-0.056,-0.142
svr_abnormal,0.023,0.142,-0.009,-0.031,-0.065
svr5_shrunk,0.015,0.107,-0.222,-0.028,-0.073


In [6]:
# By period, for the primary signal
df["period"] = pd.cut(df.date, pd.to_datetime(["2015-12-31", "2019-12-31", "2021-12-31", "2023-12-31", "2026-12-31"]),
                      labels=["2016-19", "2020-21", "2022-23", "2024-26"])
pd.DataFrame({p: daily_rank_corr(g, ["svr5"] + CONTROLS).loc["svr5", CONTROLS]
              for p, g in df.groupby("period", observed=True)}).round(3)

,2016-19,2020-21,2022-23,2024-26
ret1,0.015,0.021,0.021,0.010
ret5,0.101,0.121,0.122,0.106
log_adv20,-0.171,-0.197,-0.207,-0.304
abnormal_volume,-0.024,-0.024,-0.040,-0.030
off_exchange_share,-0.081,-0.096,-0.055,-0.058
